In [14]:
import os
import sys

current_dir = os.getcwd()
project_root = current_dir[:current_dir.find("src") - 1]
sys.path.insert(0, project_root)

from src.models.utils import *
import pandas as pd



In [15]:
plants_temperature_path = os.path.join(project_root, "data", "raw", "PlantsTemperature_View_original.csv")
p_temp_df = pd.read_csv(plants_temperature_path, encoding='utf-8')

p_temp_df["HourNo"] = p_temp_df["HourNo"].astype(int)
p_temp_df["Date"] = p_temp_df["Date"].apply(jalali_to_gregorian_fast)
p_temp_df["datetime"] = pd.to_datetime(p_temp_df["Date"]) + pd.to_timedelta(p_temp_df["HourNo"], unit='h')

is_env = p_temp_df["Code"] == "SCADAF"
tempsens_df = p_temp_df[~is_env]
tempenv_df = p_temp_df[is_env]

In [16]:
csv_semi_processed_path = os.path.join(project_root, "data", "processed", "semi_processed.csv")
semi_integrated_df = pd.read_csv(csv_semi_processed_path, encoding='utf-8')
semi_integrated_df['date'] = pd.to_datetime(semi_integrated_df['date'])
semi_integrated_df['datetime'] = semi_integrated_df['date'] + pd.to_timedelta(semi_integrated_df['hour'], unit='h')
semi_integrated_df = semi_integrated_df.rename(columns={"name": "PowerPlantName"})

# semi_integrated_df = semi_integrated_df[(semi_integrated_df["is_good_peak"] >= 3)]


In [24]:
df_merged = pd.merge(semi_integrated_df, tempsens_df, on=["datetime", "PowerPlantName"], how="inner")
print(len(df_merged))
df_merged['interval_id'] = 0


664356


In [25]:
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 664356 entries, 0 to 664355
Data columns (total 44 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   id                             664356 non-null  int64         
 1   PowerPlantName                 664356 non-null  object        
 2   code                           664356 non-null  object        
 3   date                           664356 non-null  datetime64[ns]
 4   hour                           664356 non-null  int64         
 5   temperature                    664356 non-null  float64       
 6   humidity                       664356 non-null  float64       
 7   dew                            664356 non-null  float64       
 8   apparent_temperature           664356 non-null  float64       
 9   precipitation                  664356 non-null  float64       
 10  rain                           664356 non-null  float64       
 11  

In [26]:
df_merged = df_merged.dropna()
df_merged = df_merged.rename(columns={"Value": "sen_temperature"})
df_merged = df_merged.rename(columns={"PowerPlantName": "name"})
joined_df_path = os.path.join(project_root, "data", "processed", "joined_df.csv")
df_merged.to_csv(joined_df_path)

In [27]:
df_merged

,id,name,code,date,hour,temperature,humidity,dew,apparent_temperature,precipitation,...,surface_pressure_with_5_delay,generation_with_24_delay,interval_id,PowerPlantCode,Date,Name,InsertDateTime,HourNo,sen_temperature,Code
0,104,پرند,G16,2021-03-23,6,18.328000,29.0,-0.026892,12.705196,0.0,...,891.10114,99.218530,0,104,2021-03-23,سنسور دمای نیروگاه,1401/05/05-08:59:02.86,6,15.28,TempSens
1,104,پرند,G14,2021-03-23,6,18.328000,29.0,-0.026892,12.705196,0.0,...,891.10114,109.573488,0,104,2021-03-23,سنسور دمای نیروگاه,1401/05/05-08:59:02.86,6,15.28,TempSens
2,104,پرند,G14,2021-03-23,7,19.627998,29.0,1.096538,14.612114,0.0,...,891.50470,110.059087,0,104,2021-03-23,سنسور دمای نیروگاه,1401/05/05-08:59:02.86,7,13.47,TempSens
3,104,پرند,G16,2021-03-23,7,19.627998,29.0,1.096538,14.612114,0.0,...,891.50470,99.714247,0,104,2021-03-23,سنسور دمای نیروگاه,1401/05/05-08:59:02.86,7,13.47,TempSens
4,104,پرند,G16,2021-03-23,8,20.627998,28.0,1.469439,16.130455,0.0,...,891.79870,98.095581,0,104,2021-03-23,سنسور دمای نیروگاه,1401/05/05-08:59:02.86,8,13.40,TempSens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
664351,917,حافظ,G11,2024-12-12,15,11.675000,55.0,2.930919,9.301651,0.0,...,852.01210,86.023820,0,917,2024-12-12,سنسور دمای نیروگاه,1403/09/22-05:31:17.49,15,14.48,TempSens
664352,917,حافظ,G11,2024-12-12,16,10.425000,63.0,3.679867,8.341630,0.0,...,851.69350,84.841110,0,917,2024-12-12,سنسور دمای نیروگاه,1403/09/22-05:31:17.49,16,14.00,TempSens
664353,917,حافظ,G11,2024-12-12,17,9.475000,68.0,3.861783,7.460362,0.0,...,852.03467,82.413100,0,917,2024-12-12,سنسور دمای نیروگاه,1403/09/22-05:31:17.49,17,13.39,TempSens
664354,917,حافظ,G11,2024-12-12,18,8.075000,72.0,3.330449,6.053293,0.0,...,851.83890,112.143360,0,917,2024-12-12,سنسور دمای نیروگاه,1403/09/22-05:31:17.49,18,12.19,TempSens


In [28]:
df_merged2 = pd.merge(semi_integrated_df, tempenv_df, on=["datetime", "PowerPlantName"], how="inner")
df_merged2 = df_merged2[
    ['PowerPlantName', 'code', 'date', 'hour', 'temperature', 'Value', 'generation', 'is_good_peak']].dropna()
df_merged2 = df_merged2.rename(columns={"Value": "env_temperature"})
df_merged2 = df_merged2.rename(columns={"PowerPlantName": "name"})
joined_df_path2 = os.path.join(project_root, "data", "processed", "joined_df2.csv")
df_merged2.to_csv(joined_df_path2)